# Exploring Patch-Clamp Electrophysiology Results

### **Overview**

This notebook explores the results of the patch-clamp pipeline, including:
- Session and recording metadata
- Electrophysiology features (AP threshold, input resistance, firing rate, etc.)
- Summary statistics by strain, experiment, and AP status
- Example plot visualization

#### **Setup**

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
import datajoint as dj
import pandas as pd

In [ ]:
from workflow.pipeline.patch_clamp_ephys import schema_ephys as patch_clamp
from workflow.pipeline import report

#### **1. Pipeline Coverage**

Check the number of entries in each pipeline stage.

In [ ]:
tables = [
    ("EphysExperimentsForAnalysis", patch_clamp.EphysExperimentsForAnalysis),
    ("Animals", patch_clamp.Animals),
    ("PatchCells", patch_clamp.PatchCells),
    ("EphysRecordings", patch_clamp.EphysRecordings),
    ("APandIntrinsicProperties", patch_clamp.APandIntrinsicProperties),
    ("CurrentStepPlots", patch_clamp.CurrentStepPlots),
    ("FICurvePlots", patch_clamp.FICurvePlots),
    ("VICurvePlots", patch_clamp.VICurvePlots),
    ("FirstSpikePlots", patch_clamp.FirstSpikePlots),
    ("PhasePlanes", patch_clamp.PhasePlanes),
    ("CombinedPlotsWithText", patch_clamp.CombinedPlotsWithText),
    ("AnimatedCurrentStepPlots", patch_clamp.AnimatedCurrentStepPlots),
]

pd.DataFrame(
    [(name, len(table())) for name, table in tables],
    columns=["Table", "Entries"],
)

#### **2. Experiment and Animal Overview**

List all registered experiments with their strain and recording date.

In [ ]:
(patch_clamp.Animals * patch_clamp.EphysExperimentsForAnalysis).proj(
    "strain", "date", "age", "directory"
)

#### **3. Recordings Summary**

View all recordings with their AP status.

In [ ]:
(
    patch_clamp.APandIntrinsicProperties * patch_clamp.Animals
).proj("cell", "recording", "strain", "has_ap")

#### **4. Feature Statistics**

Summarize key electrophysiology features across all recordings with action potentials.

In [ ]:
ap_query = (
    patch_clamp.APandIntrinsicProperties * patch_clamp.Animals
    & "has_ap = 'Yes'"
).proj(
    "strain", "ap_threshold", "input_resistance",
    "max_firing_rate", "f_i_curve_slope",
)

df = pd.DataFrame(ap_query.fetch(as_dict=True))
df[["ap_threshold", "input_resistance", "max_firing_rate", "f_i_curve_slope"]].describe()

#### **5. Breakdown by Strain**

Count recordings and AP status per strain.

In [ ]:
all_df = pd.DataFrame(
    (patch_clamp.APandIntrinsicProperties * patch_clamp.Animals).proj(
        "strain", "has_ap"
    ).fetch(as_dict=True)
)

all_df.groupby(["strain", "has_ap"]).size().unstack(fill_value=0)

#### **6. Feature Comparison by Strain**

Compare mean electrophysiology features across strains (AP recordings only).

In [ ]:
df.groupby("strain")[
    ["ap_threshold", "input_resistance", "max_firing_rate", "f_i_curve_slope"]
].agg(["mean", "std", "count"])

#### **7. View a Single Recording's Features**

Fetch all extracted features for a specific recording.

In [ ]:
# Pick a recording with APs to inspect
example_key = (patch_clamp.APandIntrinsicProperties & "has_ap = 'Yes'").fetch("KEY", limit=1)[0]
print(f"Example: {example_key}")

(patch_clamp.APandIntrinsicProperties & example_key)

#### **8. View Plot File Paths**

Check the generated plot files for a recording.

In [ ]:
# Current step plot paths for the example recording
(patch_clamp.CurrentStepPlots & example_key)

#### **9. Display a Plot from the Report Table**

Fetch and display an image attachment from `PatchClampReport`.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import tempfile

# Fetch a combined plot as an attachment
with tempfile.TemporaryDirectory() as tmpdir:
    filepath = (report.PatchClampReport.CombinedPlot & example_key).fetch1(
        "combined_plot", download_path=tmpdir
    )
    display(Image(filename=filepath, width=800))

#### **10. Dashboard Report Table Coverage**

In [ ]:
report_parts = [
    ("FICurve", report.PatchClampReport.FICurve),
    ("VICurve", report.PatchClampReport.VICurve),
    ("FirstSpike", report.PatchClampReport.FirstSpike),
    ("PhasePlane", report.PatchClampReport.PhasePlane),
    ("CurrentStep", report.PatchClampReport.CurrentStep),
    ("FirstSpikeDerivative", report.PatchClampReport.FirstSpikeDerivative),
    ("FirstSpikeSecondDerivative", report.PatchClampReport.FirstSpikeSecondDerivative),
    ("CombinedPlot", report.PatchClampReport.CombinedPlot),
    ("AnimatedTrace", report.PatchClampReport.AnimatedTrace),
]

pd.DataFrame(
    [(name, len(table())) for name, table in report_parts],
    columns=["Report Part Table", "Entries"],
)